# Modul 5: GPU Computing Dasar – CUDA
**Mata Kuliah:** Komputasi Paralel dan Terdistribusi  
**Kode Modul:** MOD-05-CUDA GPU  
**Dosen:** Vikri Aulia, S.Pd., M.Kom.

> ⚠️ **Catatan:** Notebook ini dijalankan di **Google Colab** (tidak ada GPU NVIDIA lokal).  
> Pastikan runtime GPU aktif: `Runtime → Change runtime type → T4 GPU`

## Langkah 1 (Dilewati – Tidak Ada GPU NVIDIA Lokal)

Karena tidak memiliki GPU NVIDIA di laptop, langkah 1 (instalasi CUDA lokal) **dilewati**.  
Seluruh eksperimen dilakukan di **Google Colab** dengan GPU T4 yang disediakan secara gratis.

Lanjut ke **Langkah 2** secara langsung.

## Langkah 2: Setup Lingkungan dan Verifikasi GPU

In [ ]:
import subprocess, os, re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def check_command(command):
    """Memeriksa apakah perintah dapat dijalankan di sistem."""
    try:
        r = subprocess.run(command, capture_output=True, text=True, shell=True)
        return r.returncode == 0, r.stdout.strip()
    except Exception:
        return False, ""

print("🔍 Memeriksa Kebutuhan Dasar CUDA Programming...\n")

# 1. Verifikasi NVIDIA Driver via nvidia-smi
has_smi, smi_out = check_command('nvidia-smi')
print('=== 1. GPU & Driver Info (nvidia-smi) ===')
if has_smi:
    print("\n".join(smi_out.split('\n')[:10]))
else:
    print("❌ Driver NVIDIA tidak terdeteksi atau perintah 'nvidia-smi' tidak ditemukan.")

print('\n' + '='*50 + '\n')

# 2. Verifikasi CUDA Compiler (nvcc)
has_nvcc, nvcc_out = check_command('nvcc --version')
print('=== 2. NVCC Version ===')
if has_nvcc:
    print(nvcc_out)
else:
    print("❌ CUDA Compiler (nvcc) TIDAK ditemukan di PATH.")

print('\n' + '='*50 + '\n')

# 3. Kesimpulan Evaluasi
if has_smi and has_nvcc:
    print("✅ SEMUA AMAN! Environment sudah siap untuk CUDA programming.")
else:
    print("⚠️ Environment BELUM siap. Pastikan runtime GPU aktif di Colab.")

In [ ]:
# Buat direktori kerja
os.makedirs('cuda_files', exist_ok=True)
print('Direktori cuda_files siap.')

# Cek compute capability GPU
r3 = subprocess.run(
    'nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader',
    shell=True, capture_output=True, text=True)
print('GPU Details:', r3.stdout.strip())

## Langkah 3: CPU Serial – Baseline

Program menerima argumen command-line untuk ukuran vektor, sehingga Python dapat menjalankan loop eksperimen dengan berbagai ukuran N.

In [ ]:
%%writefile cuda_files/vector_add_cpu.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <cstdlib>
#include <cmath>

void vector_add_cpu(float* A, float* B, float* C, int N) {
    for (int i = 0; i < N; i++) C[i] = A[i] + B[i];
}

int main(int argc, char* argv[]) {
    int N = (argc > 1) ? atoi(argv[1]) : 10000000;
    std::vector<float> A(N), B(N), C(N);
    srand(42);
    for (int i = 0; i < N; i++) {
        A[i] = (float)rand()/RAND_MAX;
        B[i] = (float)rand()/RAND_MAX;
    }
    auto t0 = std::chrono::high_resolution_clock::now();
    vector_add_cpu(A.data(), B.data(), C.data(), N);
    auto t1 = std::chrono::high_resolution_clock::now();
    double ms = std::chrono::duration<double,std::milli>(t1-t0).count();
    bool ok = true;
    for (int i=0;i<10;i++) if (fabs(C[i]-(A[i]+B[i]))>1e-6){ok=false;break;}
    float bw = (3.0f*N*sizeof(float))/(ms/1000.0f)/1e9;
    printf("N=%d TIME_MS=%.4f BW=%.4f CORRECT=%d\n",N,ms,bw,(int)ok);
    return 0;
}

In [ ]:
!g++ -O2 -std=c++11 cuda_files/vector_add_cpu.cpp -o cuda_files/vec_cpu
!./cuda_files/vec_cpu 10000000   # uji dengan 10 juta elemen

## Langkah 4: CPU dengan OpenMP (Parallel Baseline)

In [ ]:
%%writefile cuda_files/vector_add_openmp.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <cstdlib>
#include <cmath>
#include <omp.h>

int main(int argc, char* argv[]) {
    int N = (argc>1)?atoi(argv[1]):10000000;
    int T = (argc>2)?atoi(argv[2]):4;
    std::vector<float> A(N), B(N), C(N);
    srand(42);
    for (int i=0;i<N;i++){A[i]=(float)rand()/RAND_MAX;B[i]=(float)rand()/RAND_MAX;}
    omp_set_num_threads(T);
    auto t0 = std::chrono::high_resolution_clock::now();
    #pragma omp parallel for
    for (int i=0;i<N;i++) C[i]=A[i]+B[i];
    auto t1 = std::chrono::high_resolution_clock::now();
    double ms = std::chrono::duration<double,std::milli>(t1-t0).count();
    bool ok=true;
    for(int i=0;i<10;i++) if(fabs(C[i]-(A[i]+B[i]))>1e-6){ok=false;break;}
    float bw=(3.0f*N*sizeof(float))/(ms/1000.0f)/1e9;
    printf("N=%d THREADS=%d TIME_MS=%.4f BW=%.4f CORRECT=%d\n",N,T,ms,bw,(int)ok);
    return 0;
}

In [ ]:
!g++ -O2 -std=c++11 -fopenmp cuda_files/vector_add_openmp.cpp -o cuda_files/vec_omp
!./cuda_files/vec_omp 10000000 4   # 10M elemen, 4 thread

## Langkah 5: CUDA Basic – GPU Kernel

Kernel CUDA ditulis dalam file `.cu`. Setiap thread menghitung satu elemen `C[idx] = A[idx] + B[idx]`.

In [ ]:
%%writefile cuda_files/vector_add_cuda.cu
#include <iostream>
#include <cuda_runtime.h>
#include <cstdlib>
#include <cmath>

#define CHECK(call) { cudaError_t e=call; if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA error %s:%d: %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); \
    exit(1);} }

// ─── CUDA Kernel ───────────────────────────────────
__global__ void vec_add(float* A, float* B, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) C[idx] = A[idx] + B[idx];
}

int main(int argc, char* argv[]) {
    int N   = (argc>1) ? atoi(argv[1]) : 10000000;
    int TPB = (argc>2) ? atoi(argv[2]) : 256;  // threads per block
    size_t sz = N * sizeof(float);

    // ─── Alokasi Host ─────────────────────────────
    float *h_A=new float[N], *h_B=new float[N], *h_C=new float[N];
    srand(42);
    for(int i=0;i<N;i++){h_A[i]=(float)rand()/RAND_MAX;h_B[i]=(float)rand()/RAND_MAX;}

    // ─── Alokasi Device ───────────────────────────
    float *d_A,*d_B,*d_C;
    CHECK(cudaMalloc(&d_A,sz)); CHECK(cudaMalloc(&d_B,sz)); CHECK(cudaMalloc(&d_C,sz));

    // ─── CUDA Events untuk timing ─────────────────
    cudaEvent_t ev0,ev1;
    CHECK(cudaEventCreate(&ev0)); CHECK(cudaEventCreate(&ev1));
    float t_htod, t_kernel, t_dtoh;

    // HtoD transfer
    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(d_A,h_A,sz,cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_B,h_B,sz,cudaMemcpyHostToDevice));
    CHECK(cudaEventRecord(ev1)); CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_htod,ev0,ev1));

    // Kernel launch
    int blocks = (N+TPB-1)/TPB;
    CHECK(cudaEventRecord(ev0));
    vec_add<<<blocks,TPB>>>(d_A,d_B,d_C,N);
    CHECK(cudaGetLastError());
    CHECK(cudaEventRecord(ev1)); CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_kernel,ev0,ev1));

    // DtoH transfer
    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(h_C,d_C,sz,cudaMemcpyDeviceToHost));
    CHECK(cudaEventRecord(ev1)); CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_dtoh,ev0,ev1));

    float total = t_htod + t_kernel + t_dtoh;
    bool ok=true;
    for(int i=0;i<10;i++) if(fabs(h_C[i]-(h_A[i]+h_B[i]))>1e-6){ok=false;break;}
    printf("N=%d TPB=%d BLOCKS=%d HTOD=%.4f KERNEL=%.4f DTOH=%.4f TOTAL=%.4f CORRECT=%d\n",
           N,TPB,blocks,t_htod,t_kernel,t_dtoh,total,(int)ok);

    delete[]h_A;delete[]h_B;delete[]h_C;
    CHECK(cudaFree(d_A));CHECK(cudaFree(d_B));CHECK(cudaFree(d_C));
    CHECK(cudaEventDestroy(ev0));CHECK(cudaEventDestroy(ev1));
    return 0;
}

In [ ]:
!nvcc -O2 -std=c++11 cuda_files/vector_add_cuda.cu -o cuda_files/vec_cuda
!./cuda_files/vec_cuda 10000000 256   # 10M elemen, 256 thread per block

## Langkah 6: CUDA Pinned Memory – Transfer Lebih Cepat

Pinned (page-locked) memory mengunci halaman RAM sehingga transfer PCIe dapat berjalan tanpa interupsi OS, meningkatkan bandwidth transfer.

In [ ]:
%%writefile cuda_files/vector_add_pinned.cu
#include <iostream>
#include <cuda_runtime.h>
#include <cstdlib>
#include <cmath>

#define CHECK(call) { cudaError_t e=call; if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA error %s:%d: %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); \
    exit(1);} }

__global__ void vec_add(float* A, float* B, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) C[idx] = A[idx] + B[idx];
}

int main(int argc, char* argv[]) {
    int N   = (argc>1) ? atoi(argv[1]) : 10000000;
    int TPB = (argc>2) ? atoi(argv[2]) : 256;
    size_t sz = N * sizeof(float);

    // ─── Pinned memory (page-locked) ──────────────
    float *h_A, *h_B, *h_C;
    CHECK(cudaHostAlloc(&h_A, sz, cudaHostAllocDefault));
    CHECK(cudaHostAlloc(&h_B, sz, cudaHostAllocDefault));
    CHECK(cudaHostAlloc(&h_C, sz, cudaHostAllocDefault));
    srand(42);
    for(int i=0;i<N;i++){h_A[i]=(float)rand()/RAND_MAX;h_B[i]=(float)rand()/RAND_MAX;}

    float *d_A,*d_B,*d_C;
    CHECK(cudaMalloc(&d_A,sz)); CHECK(cudaMalloc(&d_B,sz)); CHECK(cudaMalloc(&d_C,sz));

    cudaEvent_t ev0,ev1;
    CHECK(cudaEventCreate(&ev0)); CHECK(cudaEventCreate(&ev1));
    float t_htod, t_kernel, t_dtoh;

    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(d_A,h_A,sz,cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_B,h_B,sz,cudaMemcpyHostToDevice));
    CHECK(cudaEventRecord(ev1)); CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_htod,ev0,ev1));

    int blocks=(N+TPB-1)/TPB;
    CHECK(cudaEventRecord(ev0));
    vec_add<<<blocks,TPB>>>(d_A,d_B,d_C,N);
    CHECK(cudaGetLastError());
    CHECK(cudaEventRecord(ev1)); CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_kernel,ev0,ev1));

    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(h_C,d_C,sz,cudaMemcpyDeviceToHost));
    CHECK(cudaEventRecord(ev1)); CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_dtoh,ev0,ev1));

    float total=t_htod+t_kernel+t_dtoh;
    bool ok=true;
    for(int i=0;i<10;i++) if(fabs(h_C[i]-(h_A[i]+h_B[i]))>1e-6){ok=false;break;}
    printf("N=%d TPB=%d HTOD=%.4f KERNEL=%.4f DTOH=%.4f TOTAL=%.4f CORRECT=%d\n",
           N,TPB,t_htod,t_kernel,t_dtoh,total,(int)ok);

    CHECK(cudaFreeHost(h_A)); CHECK(cudaFreeHost(h_B)); CHECK(cudaFreeHost(h_C));
    CHECK(cudaFree(d_A)); CHECK(cudaFree(d_B)); CHECK(cudaFree(d_C));
    CHECK(cudaEventDestroy(ev0)); CHECK(cudaEventDestroy(ev1));
    return 0;
}

In [ ]:
!nvcc -O2 -std=c++11 cuda_files/vector_add_pinned.cu -o cuda_files/vec_pinned
!./cuda_files/vec_pinned 10000000 256

## Langkah 7: Eksperimen Otomatis – Variasi Ukuran Data

Jalankan semua implementasi dengan berbagai ukuran vektor. Python mengumpulkan hasil dan membangun tabel DataFrame secara otomatis.

In [ ]:
def parse_output(out):
    d = {}
    for tok in out.split():
        if '=' in tok:
            k, v = tok.split('=', 1)
            d[k] = v
    return d

sizes = [1_000, 10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]
results = []

for N in sizes:
    row = {'N': N}

    # CPU Serial
    r = subprocess.run(f'./cuda_files/vec_cpu {N}', shell=True, capture_output=True, text=True)
    d = parse_output(r.stdout)
    row['cpu_ms'] = float(d.get('TIME_MS', 0))

    # OpenMP (4 thread)
    r = subprocess.run(f'./cuda_files/vec_omp {N} 4', shell=True, capture_output=True, text=True)
    d = parse_output(r.stdout)
    row['omp_ms'] = float(d.get('TIME_MS', 0))

    # CUDA Basic
    r = subprocess.run(f'./cuda_files/vec_cuda {N} 256', shell=True, capture_output=True, text=True)
    d = parse_output(r.stdout)
    row['cuda_htod']   = float(d.get('HTOD', 0))
    row['cuda_kernel'] = float(d.get('KERNEL', 0))
    row['cuda_dtoh']   = float(d.get('DTOH', 0))
    row['cuda_total']  = float(d.get('TOTAL', 0))

    # CUDA Pinned
    r = subprocess.run(f'./cuda_files/vec_pinned {N} 256', shell=True, capture_output=True, text=True)
    d = parse_output(r.stdout)
    row['pinned_total'] = float(d.get('TOTAL', 0))

    results.append(row)
    print(f'N={N:>12,}: cpu={row["cpu_ms"]:8.2f}ms  omp={row["omp_ms"]:7.2f}ms  '
          f'cuda={row["cuda_total"]:7.2f}ms  pinned={row["pinned_total"]:7.2f}ms')

df = pd.DataFrame(results)
print('\n=== Eksperimen selesai ===')

In [ ]:
# Hitung Speedup dan Bandwidth
df['speedup_omp']    = df['cpu_ms'] / df['omp_ms']
df['speedup_cuda']   = df['cpu_ms'] / df['cuda_total']
df['speedup_pinned'] = df['cpu_ms'] / df['pinned_total']
df['pcie_bw_cuda']   = (2 * df['N'] * 4 / 1e9) / ((df['cuda_htod'] + df['cuda_dtoh']) / 1000)  # GB/s
df['pcie_bw_pinned'] = (2 * df['N'] * 4 / 1e9) / (df['pinned_total'] / 1000)

# Cetak tabel ringkasan
print('=== Tabel Waktu Eksekusi (ms) ===')
print(df[['N','cpu_ms','omp_ms','cuda_total','pinned_total']].to_string(index=False))
print('\n=== Tabel Speedup ===')
print(df[['N','speedup_omp','speedup_cuda','speedup_pinned']].to_string(index=False))
print('\n=== Tabel CUDA Breakdown (ms) ===')
print(df[['N','cuda_htod','cuda_kernel','cuda_dtoh','cuda_total']].to_string(index=False))

## Langkah 8: Visualisasi Hasil

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Grafik 1: Waktu Eksekusi (log scale)
ax = axes[0, 0]
ax.loglog(df['N'], df['cpu_ms'],     'b-o', label='CPU Serial',  linewidth=2, markersize=7)
ax.loglog(df['N'], df['omp_ms'],     'g-s', label='CPU OpenMP',  linewidth=2, markersize=7)
ax.loglog(df['N'], df['cuda_total'], 'r-^', label='CUDA Basic',  linewidth=2, markersize=7)
ax.loglog(df['N'], df['pinned_total'],'m-D', label='CUDA Pinned', linewidth=2, markersize=7)
ax.set_xlabel('Ukuran Vektor (N)')
ax.set_ylabel('Waktu (ms)')
ax.set_title('Waktu Eksekusi vs Ukuran Vektor (log-log)')
ax.legend()
ax.grid(True, alpha=0.4)

# Grafik 2: Speedup vs CPU Serial
ax = axes[0, 1]
ax.semilogx(df['N'], df['speedup_omp'],    'g-s', label='OpenMP',      linewidth=2, markersize=7)
ax.semilogx(df['N'], df['speedup_cuda'],   'r-^', label='CUDA Basic',  linewidth=2, markersize=7)
ax.semilogx(df['N'], df['speedup_pinned'], 'm-D', label='CUDA Pinned', linewidth=2, markersize=7)
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7, label='Break-even (1×)')
ax.set_xlabel('Ukuran Vektor (N)')
ax.set_ylabel('Speedup vs CPU Serial')
ax.set_title('Speedup vs Ukuran Vektor')
ax.legend()
ax.grid(True, alpha=0.4)

# Grafik 3: CUDA Breakdown (10M elemen)
ax = axes[1, 0]
row10m = df[df['N'] == 10_000_000].iloc[0]
labels_b = ['HtoD Transfer', 'Kernel Exec', 'DtoH Transfer']
vals_b   = [row10m['cuda_htod'], row10m['cuda_kernel'], row10m['cuda_dtoh']]
colors_b = ['#E74C3C', '#2ECC71', '#3498DB']
ax.pie(vals_b, labels=labels_b, colors=colors_b,
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
ax.set_title(f'Breakdown Waktu CUDA (N=10M)\nTotal: {row10m["cuda_total"]:.2f}ms')

# Grafik 4: Bandwidth PCIe
ax = axes[1, 1]
ax.semilogx(df['N'], df['pcie_bw_cuda'],   'r-^', label='CUDA Basic',  linewidth=2, markersize=7)
ax.semilogx(df['N'], df['pcie_bw_pinned'], 'm-D', label='CUDA Pinned', linewidth=2, markersize=7)
ax.axhline(y=16, color='gray', linestyle='--', alpha=0.7, label='PCIe 3.0 Teoritis (16 GB/s)')
ax.set_xlabel('Ukuran Vektor (N)')
ax.set_ylabel('Bandwidth (GB/s)')
ax.set_title('Bandwidth PCIe vs Ukuran Data')
ax.legend()
ax.grid(True, alpha=0.4)

plt.suptitle('Modul 5: CUDA GPU Computing – Performance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cuda_files/performance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan ke cuda_files/performance_analysis.png')

## Langkah 9: Eksperimen Thread per Block

In [ ]:
tpb_configs = [32, 64, 128, 256, 512, 1024]
N_test = 10_000_000
tpb_results = []

for tpb in tpb_configs:
    r = subprocess.run(f'./cuda_files/vec_cuda {N_test} {tpb}',
                       shell=True, capture_output=True, text=True)
    d = parse_output(r.stdout)
    tpb_results.append({
        'ThreadsPerBlock': tpb,
        'Blocks'         : int(d.get('BLOCKS', 0)),
        'Kernel_ms'      : float(d.get('KERNEL', 0)),
        'Total_ms'       : float(d.get('TOTAL', 0))
    })
    print(f'TPB={tpb:5d}: kernel={float(d.get("KERNEL",0)):.4f}ms  total={float(d.get("TOTAL",0)):.4f}ms')

df_tpb = pd.DataFrame(tpb_results)
print('\n=== Tabel Thread per Block ===')
print(df_tpb.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(tpb_configs)), df_tpb['Kernel_ms'], color='steelblue', alpha=0.8, edgecolor='navy')
ax.set_xticks(range(len(tpb_configs)))
ax.set_xticklabels([str(t) for t in tpb_configs])
ax.set_xlabel('Thread per Block')
ax.set_ylabel('Kernel Time (ms)')
ax.set_title(f'Kernel Time vs Thread per Block (N={N_test:,})')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('cuda_files/tpb_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan ke cuda_files/tpb_analysis.png')

## Ringkasan dan Analisis

### Kesimpulan Eksperimen

1. **Break-even point GPU vs CPU**: GPU mulai unggul pada N ≥ 1.000.000 elemen. Pada data kecil, overhead transfer PCIe mendominasi.

2. **Komposisi waktu CUDA**: Transfer HtoD + DtoH memakan porsi besar untuk N kecil. Makin besar N, proporsi kernel semakin dominan.

3. **Pinned Memory**: Memberikan peningkatan bandwidth transfer karena RAM tidak di-swap oleh OS, sehingga DMA (Direct Memory Access) lebih efisien.

4. **Thread per Block optimal**: Umumnya 256 atau 512 thread/block memberikan performa terbaik karena sesuai dengan warp size (32) dan kapasitas SM pada GPU NVIDIA.